# Visualization and Analysis
This notebook creates comprehensive visualizations of the HMM models, data, and results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Load Models and Results

In [ ]:
# Load trained models
models = joblib.load('../models/hmm_models.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Load evaluation results
predictions_df = pd.read_csv('../results/predictions/unseen_test_predictions.csv')
confusion_matrix_df = pd.read_csv('../results/metrics/confusion_matrix_unseen.csv', index_col=0)

# Load training features
training_features = pd.read_csv('../processed_data/training_features.csv')

print(f"Loaded {len(models)} HMM models")
print(f"Activities: {list(models.keys())}")
print(f"Predictions: {len(predictions_df)} samples")
print(f"Training features: {training_features.shape}")

## 2. HMM Model Visualizations
### Transition Probability Matrices

In [ ]:
# Visualize transition matrices for all activities
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (activity, model) in enumerate(models.items()):
    # Get transition matrix
    trans_matrix = model.transmat_
    
    # Plot heatmap
    sns.heatmap(trans_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                ax=axes[idx], cbar_kws={'label': 'Probability'},
                xticklabels=[f'S{i}' for i in range(trans_matrix.shape[0])],
                yticklabels=[f'S{i}' for i in range(trans_matrix.shape[0])])
    axes[idx].set_title(f'{activity.upper()} - Transition Matrix', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('To State')
    axes[idx].set_ylabel('From State')

plt.tight_layout()
plt.savefig('../results/figures/hmm_transition_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Transition matrices visualization saved")

### Initial State Probabilities

In [ ]:
# Plot initial state probabilities
fig, ax = plt.subplots(figsize=(12, 6))

activities = list(models.keys())
x = np.arange(len(activities))
width = 0.2

for i in range(4):  # 4 hidden states
    probs = [models[act].startprob_[i] for act in activities]
    ax.bar(x + i*width, probs, width, label=f'State {i}')

ax.set_xlabel('Activity', fontsize=12)
ax.set_ylabel('Initial Probability', fontsize=12)
ax.set_title('Initial State Probabilities by Activity', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(activities)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/initial_state_probabilities.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Initial state probabilities visualization saved")

## 3. Feature Distributions
### Compare feature distributions across activities

In [ ]:
# Select key features to visualize
key_features = ['accel_x_mean', 'accel_y_mean', 'accel_z_mean',
                'accel_x_std', 'accel_y_std', 'accel_z_std',
                'gyro_x_mean', 'gyro_y_mean', 'gyro_z_mean',
                'accel_sma']

# Create violin plots
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    for activity in training_features['activity'].unique():
        data = training_features[training_features['activity'] == activity][feature]
        axes[idx].violinplot([data], positions=[list(training_features['activity'].unique()).index(activity)],
                            widths=0.7, showmeans=True, showextrema=True)
    
    axes[idx].set_title(f'{feature}', fontsize=11, fontweight='bold')
    axes[idx].set_xticks(range(len(training_features['activity'].unique())))
    axes[idx].set_xticklabels(training_features['activity'].unique(), rotation=45)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylabel('Value')

plt.suptitle('Feature Distributions by Activity', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature distributions visualization saved")

### Feature Correlation Heatmap

In [ ]:
# Calculate correlation matrix for numerical features
numerical_features = training_features.select_dtypes(include=[np.number])
correlation_matrix = numerical_features.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature correlation heatmap saved")

## 4. Model Performance Visualizations
### Accuracy by Activity

In [ ]:
# Calculate per-activity accuracy
activity_accuracy = []
for activity in predictions_df['true_activity'].unique():
    mask = predictions_df['true_activity'] == activity
    correct = (predictions_df[mask]['true_activity'] == predictions_df[mask]['predicted_activity']).sum()
    total = mask.sum()
    accuracy = correct / total if total > 0 else 0
    activity_accuracy.append({'activity': activity, 'accuracy': accuracy, 'samples': total})

accuracy_df = pd.DataFrame(activity_accuracy)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(accuracy_df['activity'], accuracy_df['accuracy'], color='steelblue', alpha=0.8)

# Add value labels on bars
for i, (bar, row) in enumerate(zip(bars, accuracy_df.itertuples())):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1%}\n(n={row.samples})',
            ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Activity', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy by Activity', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.axhline(y=accuracy_df['accuracy'].mean(), color='red', linestyle='--', 
           label=f'Mean: {accuracy_df["accuracy"].mean():.1%}', linewidth=2)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/figures/accuracy_by_activity.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Accuracy by activity visualization saved")

### Prediction Confidence Distribution

In [ ]:
# Plot confidence distribution for correct vs incorrect predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Separate correct and incorrect predictions
correct = predictions_df[predictions_df['correct']]['confidence']
incorrect = predictions_df[~predictions_df['correct']]['confidence']

# Histogram comparison
axes[0].hist(correct, bins=30, alpha=0.6, label=f'Correct (n={len(correct)})', color='green')
axes[0].hist(incorrect, bins=30, alpha=0.6, label=f'Incorrect (n={len(incorrect)})', color='red')
axes[0].set_xlabel('Confidence Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot by activity
confidence_by_activity = predictions_df.groupby('true_activity')['confidence'].apply(list)
axes[1].boxplot(confidence_by_activity.values, labels=confidence_by_activity.index)
axes[1].set_xlabel('Activity', fontsize=12)
axes[1].set_ylabel('Confidence Score', fontsize=12)
axes[1].set_title('Confidence by Activity', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/figures/confidence_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confidence distribution visualization saved")

## 5. Sample Sensor Data Visualization
### Visualize raw sensor data for each activity

In [ ]:
# Load sample raw sensor data for each activity
data_dir = Path('../data/ange')
activities_to_plot = ['jumping', 'walking', 'standing', 'still']

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for idx, activity in enumerate(activities_to_plot):
    activity_dir = data_dir / activity
    
    # Get first recording
    recordings = sorted([d for d in activity_dir.iterdir() if d.is_dir()])
    if recordings:
        recording = recordings[0]
        
        # Load accelerometer and gyroscope data
        accel_file = recording / 'Accelerometer.csv'
        gyro_file = recording / 'Gyroscope.csv'
        
        if accel_file.exists() and gyro_file.exists():
            accel = pd.read_csv(accel_file)
            gyro = pd.read_csv(gyro_file)
            
            # Take first 200 samples
            accel = accel.head(200)
            gyro = gyro.head(200)
            
            # Plot accelerometer
            axes[idx, 0].plot(accel['x'], label='X', alpha=0.7)
            axes[idx, 0].plot(accel['y'], label='Y', alpha=0.7)
            axes[idx, 0].plot(accel['z'], label='Z', alpha=0.7)
            axes[idx, 0].set_title(f'{activity.upper()} - Accelerometer', fontweight='bold')
            axes[idx, 0].set_ylabel('Acceleration (m/s²)')
            axes[idx, 0].legend(loc='upper right')
            axes[idx, 0].grid(True, alpha=0.3)
            
            # Plot gyroscope
            axes[idx, 1].plot(gyro['x'], label='X', alpha=0.7)
            axes[idx, 1].plot(gyro['y'], label='Y', alpha=0.7)
            axes[idx, 1].plot(gyro['z'], label='Z', alpha=0.7)
            axes[idx, 1].set_title(f'{activity.upper()} - Gyroscope', fontweight='bold')
            axes[idx, 1].set_ylabel('Angular Velocity (rad/s)')
            axes[idx, 1].legend(loc='upper right')
            axes[idx, 1].grid(True, alpha=0.3)

# Set common x-label for bottom plots
axes[3, 0].set_xlabel('Sample Number')
axes[3, 1].set_xlabel('Sample Number')

plt.suptitle('Sample Sensor Readings by Activity', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../results/figures/sample_sensor_data.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Sample sensor data visualization saved")

## 6. Confusion Matrix Analysis
### Detailed confusion matrix with percentages

In [ ]:
# Calculate percentage confusion matrix
cm_percent = confusion_matrix_df.div(confusion_matrix_df.sum(axis=1), axis=0) * 100

# Create figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(confusion_matrix_df, annot=True, fmt='g', cmap='Blues',
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Activity')
axes[0].set_ylabel('True Activity')

# Percentages
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Greens',
            ax=axes[1], cbar_kws={'label': 'Percentage'})
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Activity')
axes[1].set_ylabel('True Activity')

plt.tight_layout()
plt.savefig('../results/figures/confusion_matrix_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Detailed confusion matrix visualization saved")

## 7. Summary Statistics

In [ ]:
# Calculate and display summary statistics
overall_accuracy = predictions_df['correct'].mean()

print("="*60)
print("VISUALIZATION SUMMARY")
print("="*60)
print(f"\nOverall Model Accuracy: {overall_accuracy:.2%}")
print(f"Total Test Samples: {len(predictions_df)}")
print(f"\nPer-Activity Accuracy:")
for _, row in accuracy_df.iterrows():
    print(f"  {row['activity']:10s}: {row['accuracy']:.2%} ({row['samples']} samples)")

print(f"\nAverage Confidence (Correct): {predictions_df[predictions_df['correct']]['confidence'].mean():.2f}")
print(f"Average Confidence (Incorrect): {predictions_df[~predictions_df['correct']]['confidence'].mean():.2f}")

print("\n" + "="*60)
print("GENERATED VISUALIZATIONS")
print("="*60)
print("All visualizations saved to: results/figures/")
print("  1. hmm_transition_matrices.png")
print("  2. initial_state_probabilities.png")
print("  3. feature_distributions.png")
print("  4. feature_correlation.png")
print("  5. accuracy_by_activity.png")
print("  6. confidence_distribution.png")
print("  7. sample_sensor_data.png")
print("  8. confusion_matrix_detailed.png")
print("="*60)

## Summary

This visualization notebook provides comprehensive analysis of:
1. ✅ HMM model structure (transition matrices, initial states)
2. ✅ Feature distributions and correlations
3. ✅ Model performance metrics
4. ✅ Prediction confidence analysis
5. ✅ Raw sensor data patterns
6. ✅ Detailed confusion matrix analysis

All visualizations have been saved to the `results/figures/` directory for documentation and reporting.